In [0]:
%sql
CREATE TABLE IF NOT EXISTS retail_catalog.silver.dim_customer (
    CustomerSK BIGINT GENERATED ALWAYS AS IDENTITY,
    CustomerID INT,
    CustomerName STRING,
    Email STRING,
    City STRING,
    Address STRING,
    StartDate DATE,
    EndDate DATE,
    IsActive INT
)
USING DELTA;

In [0]:
%sql
-- SCD Type 2 Implementation for Customer Dimension
-- When a unique record arrives: IsActive = 1
-- When a duplicate arrives with changes: old record gets IsActive = 0, new record gets IsActive = 1
-- Both old and new records remain in the table (history preserved)

-- Step 1: Mark existing active records as inactive when changes are detected
MERGE INTO retail_catalog.silver.dim_customer target
USING (
    SELECT
        CustomerID,
        CustomerName,
        Email,
        City,
        Address
    FROM (
        SELECT
            CustomerID,
            INITCAP(TRIM(CustomerName)) AS CustomerName,
            LOWER(Email) AS Email,
            TRIM(City) AS City,
            TRIM(Address) AS Address,
            ROW_NUMBER() OVER (PARTITION BY CustomerID ORDER BY LastUpdated DESC) AS rn
        FROM retail_catalog.bronze.customers_raw
    )
    WHERE rn = 1
) source
ON target.CustomerID = source.CustomerID
   AND target.IsActive = 1  -- Only match currently active records

-- When matched and data has changed: mark old record as inactive (IsActive = 0)
WHEN MATCHED AND (
    target.City <> source.City OR
    target.Address <> source.Address OR
    target.CustomerName <> source.CustomerName OR
    target.Email <> source.Email
)
THEN UPDATE SET
    target.EndDate = CURRENT_DATE(),
    target.IsActive = 0  -- Mark old record as inactive

-- When not matched (new unique customer): insert with IsActive = 1
WHEN NOT MATCHED
THEN INSERT (
    CustomerID,
    CustomerName,
    Email,
    City,
    Address,
    StartDate,
    EndDate,
    IsActive
)
VALUES (
    source.CustomerID,
    source.CustomerName,
    source.Email,
    source.City,
    source.Address,
    CURRENT_DATE(),
    DATE('9999-12-31'),
    1  -- New unique record gets IsActive = 1
);

-- Step 2: Insert new active versions for customers whose records were marked inactive
-- This creates the new record with IsActive = 1 while keeping the old record with IsActive = 0
INSERT INTO retail_catalog.silver.dim_customer (
    CustomerID,
    CustomerName,
    Email,
    City,
    Address,
    StartDate,
    EndDate,
    IsActive
)
SELECT
    source.CustomerID,
    source.CustomerName,
    source.Email,
    source.City,
    source.Address,
    CURRENT_DATE() AS StartDate,
    DATE('9999-12-31') AS EndDate,
    1 AS IsActive  -- New version gets IsActive = 1
FROM (
    SELECT
        CustomerID,
        INITCAP(TRIM(CustomerName)) AS CustomerName,
        LOWER(Email) AS Email,
        TRIM(City) AS City,
        TRIM(Address) AS Address,
        ROW_NUMBER() OVER (PARTITION BY CustomerID ORDER BY LastUpdated DESC) AS rn
    FROM retail_catalog.bronze.customers_raw
) source
WHERE rn = 1
  -- Only insert if there's no active record with the same data
  AND NOT EXISTS (
      SELECT 1 
      FROM retail_catalog.silver.dim_customer target
      WHERE target.CustomerID = source.CustomerID
        AND target.IsActive = 1
        AND target.City = source.City
        AND target.Address = source.Address
        AND target.CustomerName = source.CustomerName
        AND target.Email = source.Email
  )
  -- Only insert if the customer exists (was marked inactive in step 1)
  AND EXISTS (
      SELECT 1
      FROM retail_catalog.silver.dim_customer target
      WHERE target.CustomerID = source.CustomerID
  );

In [0]:
%sql
-- Step 1: Mark old records as inactive when changes are detected
MERGE INTO retail_catalog.silver.dim_customer target
USING (
    SELECT
        CustomerID,
        CustomerName,
        Email,
        City,
        Address
    FROM (
        SELECT
            CustomerID,
            INITCAP(TRIM(CustomerName)) AS CustomerName,
            LOWER(Email) AS Email,
            TRIM(City) AS City,
            TRIM(Address) AS Address,
            ROW_NUMBER() OVER (PARTITION BY CustomerID ORDER BY LastUpdated DESC) AS rn
        FROM retail_catalog.bronze.customers_raw
    )
    WHERE rn = 1
) source
ON target.CustomerID = source.CustomerID
   AND target.IsActive = 1

WHEN MATCHED AND (
    target.City <> source.City OR
    target.Address <> source.Address
)
THEN UPDATE SET
    target.EndDate = CURRENT_DATE(),
    target.IsActive = 0

WHEN NOT MATCHED
THEN INSERT (
    CustomerID,
    CustomerName,
    Email,
    City,
    Address,
    StartDate,
    EndDate,
    IsActive
)
VALUES (
    source.CustomerID,
    source.CustomerName,
    source.Email,
    source.City,
    source.Address,
    CURRENT_DATE(),
    DATE('9999-12-31'),
    1
);

In [0]:
%sql
select * from retail_catalog.silver.dim_customer where IsActive=0 limit 10;

In [0]:
%sql
-- Step 2: Insert new active records for customers with changes
INSERT INTO retail_catalog.silver.dim_customer (
    CustomerID,
    CustomerName,
    Email,
    City,
    Address,
    StartDate,
    EndDate,
    IsActive
)
SELECT
    source.CustomerID,
    source.CustomerName,
    source.Email,
    source.City,
    source.Address,
    CURRENT_DATE() AS StartDate,
    DATE('9999-12-31') AS EndDate,
    1 AS IsActive
FROM (
    SELECT
        CustomerID,
        INITCAP(TRIM(CustomerName)) AS CustomerName,
        LOWER(Email) AS Email,
        TRIM(City) AS City,
        TRIM(Address) AS Address,
        ROW_NUMBER() OVER (PARTITION BY CustomerID ORDER BY LastUpdated DESC) AS rn
    FROM retail_catalog.bronze.customers_raw
) source
WHERE rn = 1
  AND NOT EXISTS (
      SELECT 1 
      FROM retail_catalog.silver.dim_customer target
      WHERE target.CustomerID = source.CustomerID
        AND target.IsActive = 1
        AND target.City = source.City
        AND target.Address = source.Address
  );

In [0]:
%sql
select * from retail_catalog.silver.dim_customer 

In [0]:
%sql
CREATE TABLE IF NOT EXISTS retail_catalog.silver.dim_product
USING DELTA AS
SELECT
    monotonically_increasing_id() AS ProductSK,
    ProductID,
    TRIM(ProductName) AS ProductName,
    TRIM(Category) AS Category,
    UnitPrice,
    CURRENT_DATE() AS EffectiveDate
FROM retail_catalog.bronze.products_raw;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS retail_catalog.silver.dim_store
USING DELTA AS
SELECT
    monotonically_increasing_id() AS StoreSK,
    StoreID,
    TRIM(StoreName) AS StoreName,
    TRIM(Region) AS Region
FROM retail_catalog.bronze.stores_raw;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS retail_catalog.silver.fact_sales
USING DELTA AS
SELECT
    monotonically_increasing_id() AS SalesSK,
    s.TransactionID,
    c.CustomerSK,
    p.ProductSK,
    st.StoreSK,
    s.Quantity,
    (s.Quantity * p.UnitPrice) AS Amount,
    DATE(s.TxnDate) AS TxnDate
FROM retail_catalog.bronze.sales_raw s

LEFT JOIN retail_catalog.silver.dim_customer c
    ON s.CustomerID = c.CustomerID AND c.IsActive = 1

LEFT JOIN retail_catalog.silver.dim_product p
    ON s.ProductID = p.ProductID

LEFT JOIN retail_catalog.silver.dim_store st
    ON s.StoreID = st.StoreID;

In [0]:
%sql
SELECT CustomerID, COUNT(*)
FROM retail_catalog.silver.dim_customer
WHERE IsActive = 1
GROUP BY CustomerID
HAVING COUNT(*) > 1;

In [0]:
%sql
SELECT *
FROM retail_catalog.silver.fact_sales
WHERE CustomerSK IS NULL
   OR ProductSK IS NULL
   OR StoreSK IS NULL;

In [0]:
%sql
SELECT *
FROM retail_catalog.silver.fact_sales